In [1]:
! pip install python-dotenv pymongo

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from pymongo import MongoClient
from bson import ObjectId

load_dotenv()

DB_NAME = "drug-sentencing-predictor"
EXTRACTED_FEATURES_COLLECTION_NAME = "llm-extracted-features"
VERIFIED_FEATURES_COLLECTION_NAME = "verified-features"

uri = os.getenv("DB_MONGODB_URI")
client = MongoClient(uri)
database = client.get_database(DB_NAME)
extracted_features_collection = database.get_collection(EXTRACTED_FEATURES_COLLECTION_NAME)
verified_features_collection = database.get_collection(VERIFIED_FEATURES_COLLECTION_NAME)



def _normalize_bytes(value):
    if isinstance(value, ObjectId):
        return str(value)
    return value

def _add_projection_stage(pipeline):
    return pipeline + [
		 {
			"$lookup": {
			"from": "user",
			"localField": "verified_by",
			"foreignField": "_id",
			"as": "assigned_user"
			}
		},
		{
			"$project": {
			"neutral_citation": "$judgement.neutral_citation",
			"charge_number": "$judgement.charges.charge_no",
			"defendant_id": "$judgement.charges.defendants_of_charge.defendant_id",
			"assigned_username": {"$arrayElemAt": ["$assigned_user.username", 0]},
			"assigned_name": {"$arrayElemAt": ["$assigned_user.name", 0]},
			"assigned_email": {"$arrayElemAt": ["$assigned_user.email", 0]},
			"_id": 0
			}
		}
	]

def get_flagged_cases(pipeline):
    """Helper function to run the MongoDB aggregation and return a pandas DataFrame."""
    cursor = verified_features_collection.aggregate(_add_projection_stage(pipeline))
    # df = pd.DataFrame(list(cursor))
    data = list(cursor)
    # df = df.map(_normalize_bytes) 
    return data

def _add_defendant_projection_stage(pipeline):
    pipeline = pipeline + [
		 {
			"$lookup": {
			"from": "user",
			"localField": "verified_by",
			"foreignField": "_id",
			"as": "assigned_user"
			}
		},
		{
			"$project": {
				"neutral_citation": "$judgement.neutral_citation",
				"defendant_id": "$defendants.defendants.defendant_id",
				"defendant_name": "$defendants.defendants.defendant_name.name",
				"assigned_username": {"$arrayElemAt": ["$assigned_user.username", 0]},
				"assigned_name": {"$arrayElemAt": ["$assigned_user.name", 0]},
				"assigned_email": {"$arrayElemAt": ["$assigned_user.email", 0]},
				"_id": 0
			}
		}
	]
    

    return pipeline

def get_defendant_flagged_cases(pipeline):
    """Helper function to run defendant-level aggregations."""
    cursor = verified_features_collection.aggregate(_add_defendant_projection_stage(pipeline))
    data = list(cursor)
    return data


In [3]:
def _add_trials_projection_stage(pipeline):
    pipeline = pipeline + [
        {
            "$lookup": {
                "from": "user",
                "localField": "verified_by",
                "foreignField": "_id",
                "as": "assigned_user"
            }
        },
        {
            "$project": {
                "neutral_citation": "$judgement.neutral_citation",
                "charge_number": "$trials.trials.charge_type.charge_no",
                "defendant_id": "$trials.trials.charge_type.defendant_id",
                "assigned_username": {"$arrayElemAt": ["$assigned_user.username", 0]},
                "assigned_name": {"$arrayElemAt": ["$assigned_user.name", 0]},
                "assigned_email": {"$arrayElemAt": ["$assigned_user.email", 0]},
                "_id": 0
            }
        },
    ]

    return pipeline

def get_trials_flagged_cases(pipeline):
    """Helper function to run trial-level aggregations."""
    cursor = verified_features_collection.aggregate(_add_trials_projection_stage(pipeline))
    data = list(cursor)
    return data

In [4]:
tables = {}

## charges
### place_of_offence
- subDistrict: Not specified

In [5]:
pipeline_place_of_offence = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$match": {
      "judgement.charges.place_of_offence.subDistrict": { "$in": [None, ""] }
    }
  }
]

data = get_flagged_cases(pipeline_place_of_offence)
tables["place_of_offence.1"] = data
data

[{'neutral_citation': '[2021] HKCFI 392',
  'charge_number': 2,
  'defendant_id': [1],
  'assigned_username': 'user.30',
  'assigned_name': 'User 30 - Fung Chun Hei, Justin',
  'assigned_email': 'justinf@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2837',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.11',
  'assigned_name': 'User 11 - Kwai Hoi Yan, Hayley',
  'assigned_email': 'hayleyk@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 1718',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.16',
  'assigned_name': 'User 16 - Lee Hwang, Joseph',
  'assigned_email': 'u3632543@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 1719',
  'charge_number': 2,
  'defendant_id': [1],
  'assigned_username': 'user.16',
  'assigned_name': 'User 16 - Lee Hwang, Joseph',
  'assigned_email': 'u3632543@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 3320',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user

### defendants_of_charge
#### trafficking_mode

In [6]:
pipeline_trafficking_mode_drug_houses = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$unwind": {"path": "$judgement.charges.defendants_of_charge"}
  },
  {
    "$match": {
      "judgement.charges.defendants_of_charge.trafficking_mode.mode": "Drug houses"
    }
  },
]

data = get_flagged_cases(pipeline_trafficking_mode_drug_houses)
tables["trafficking_mode.1"] = data
data

[{'neutral_citation': '[2021] HKCFI 2628',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'removed',
  'assigned_name': 'User - Removed',
  'assigned_email': 'removed'},
 {'neutral_citation': '[2021] HKDC 909',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.20',
  'assigned_name': 'User 20 - Cheung Ka-Lun, Karen',
  'assigned_email': 'karen.c13@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 742',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.32',
  'assigned_name': 'User 32 - Kulpriya Lam',
  'assigned_email': 'kulpriya@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 734',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.3',
  'assigned_name': 'User 3 - Wong Man Hei, Pheobe',
  'assigned_email': 'wongmanheipheobe@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 775',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.3',
  'assigned_name': 'User 3 - Wong Man 

In [7]:
pipeline_trafficking_mode_mule_and_courier = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$match": {
      "$and": [
        {
          "judgement.charges.defendants_of_charge": {
            "$elemMatch": {"trafficking_mode.mode": "Mule trafficking"}
          }
        },
        {
          "judgement.charges.defendants_of_charge": {
            "$elemMatch": {"trafficking_mode.mode": "Courier delivery"}
          }
        }
      ]
    }
  },
]

data = get_flagged_cases(pipeline_trafficking_mode_mule_and_courier)
tables["trafficking_mode.2"] = data
data

[{'neutral_citation': '[2025] HKCFI 4197',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.34',
  'assigned_name': 'User 34 - Zhang Yuen Ka, Winnie',
  'assigned_email': 'winniezyk@connect.hku.hk'},
 {'neutral_citation': '[2025] HKCFI 3255',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.29',
  'assigned_name': 'User 29 - Lau Wan',
  'assigned_email': 'u3638683@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2880',
  'charge_number': 1,
  'defendant_id': [1, 2],
  'assigned_username': 'user.24',
  'assigned_name': 'User 24 - Lee Hiu Wa',
  'assigned_email': 'dorcasyy@connect.hku.hk'},
 {'neutral_citation': '[2025] HKDC 1614',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.13',
  'assigned_name': 'User 13 - Shum Pak Yin',
  'assigned_email': 'pyshum05@connect.hku.hk'},
 {'neutral_citation': '[2025] HKCFI 3027',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.40',
  'assigned_n

In [8]:
pipeline_trafficking_mode_repackage_and_courier = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$match": {
      "$and": [
        {
          "judgement.charges.defendants_of_charge": {
            "$elemMatch": {"trafficking_mode.mode": "Drug repackaging or storage"}
          }
        },
        {
          "judgement.charges.defendants_of_charge": {
            "$elemMatch": {"trafficking_mode.mode": "Courier delivery"}
          }
        }
      ]
    }
  },
]

data = get_flagged_cases(pipeline_trafficking_mode_repackage_and_courier)
tables["trafficking_mode.3"] = data
data

[{'neutral_citation': '[2022] HKCFI 205',
  'charge_number': 1,
  'defendant_id': [1, 2],
  'assigned_username': 'user.14',
  'assigned_name': 'User 14 - Mak Ho Lam, Clarence',
  'assigned_email': 'cmak1@connect.hku.hk'},
 {'neutral_citation': '[2022] HKCFI 205',
  'charge_number': 3,
  'defendant_id': [1, 2],
  'assigned_username': 'user.14',
  'assigned_name': 'User 14 - Mak Ho Lam, Clarence',
  'assigned_email': 'cmak1@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 41',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.9',
  'assigned_name': 'User 9 - Xu Wenxuan',
  'assigned_email': 'jonwardxu@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 986',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.17',
  'assigned_name': 'User 17 - He Fangyi, Joanna',
  'assigned_email': 'hphiii3355@connect.hku.hk'},
 {'neutral_citation': '[2024] HKCFI 1501',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.14',
  'as

In [9]:
pipeline_place_public_transport_and_courier = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
#   {
#     "$unwind": {"path": "$judgement.charges.defendants_of_charge"}
#   },
  {
    "$match": {
      "judgement.charges.place_of_offence.nature": "Public transport",
      "judgement.charges.defendants_of_charge.trafficking_mode.mode": "Courier delivery"
    }
  },
]

data = get_flagged_cases(pipeline_place_public_transport_and_courier)
tables["place_of_offence.4"] = data
data

[{'neutral_citation': '[2021] HKDC 801',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.24',
  'assigned_name': 'User 24 - Lee Hiu Wa',
  'assigned_email': 'dorcasyy@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 538',
  'charge_number': 2,
  'defendant_id': [1],
  'assigned_username': 'user.4',
  'assigned_name': 'User 4 - Chloe Chow',
  'assigned_email': 'chloe1@connect.hku.hk'},
 {'neutral_citation': '[2023] HKDC 1551',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.28',
  'assigned_name': 'User 28 - Wong Tin Wing, Winnie',
  'assigned_email': 'winnie43@connect.hku.hk'},
 {'neutral_citation': '[2025] HKDC 1727',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.27',
  'assigned_name': 'User 27 - Wong Ling Yan, Ada',
  'assigned_email': 'lywong24@connect.hku.hk'},
 {'neutral_citation': '[2022] HKDC 495',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.29',
  'assigned_name': 

In [10]:
pipeline_place_public_transport_and_possession = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$unwind": {"path": "$judgement.charges.defendants_of_charge"}
  },
  {
    "$match": {
      "judgement.charges.place_of_offence.nature": "Public transport",
      "judgement.charges.defendants_of_charge.trafficking_mode.mode": "Possession for trafficking"
    }
  },
]

data = get_flagged_cases(pipeline_place_public_transport_and_possession)
tables["trafficking_mode.5"] = data
data

[{'neutral_citation': '[2023] HKDC 41',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.28',
  'assigned_name': 'User 28 - Wong Tin Wing, Winnie',
  'assigned_email': 'winnie43@connect.hku.hk'},
 {'neutral_citation': '[2023] HKDC 292',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.11',
  'assigned_name': 'User 11 - Kwai Hoi Yan, Hayley',
  'assigned_email': 'hayleyk@connect.hku.hk'},
 {'neutral_citation': '[2024] HKDC 1883',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.14',
  'assigned_name': 'User 14 - Mak Ho Lam, Clarence',
  'assigned_email': 'cmak1@connect.hku.hk'},
 {'neutral_citation': '[2024] HKDC 1723',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.30',
  'assigned_name': 'User 30 - Fung Chun Hei, Justin',
  'assigned_email': 'justinf@connect.hku.hk'},
 {'neutral_citation': '[2025] HKCFI 6390',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.20',
  'ass

In [11]:
pipeline_place_private_vehicle_and_courier = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$unwind": {"path": "$judgement.charges.defendants_of_charge"}
  },
  {
    "$match": {
      "judgement.charges.place_of_offence.nature": "Private vehicle",
      "judgement.charges.defendants_of_charge.trafficking_mode.mode": "Courier delivery"
    }
  },
]

data = get_flagged_cases(pipeline_place_private_vehicle_and_courier)
tables["trafficking_mode.6"] = data
data

[{'neutral_citation': '[2021] HKDC 1075',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.35',
  'assigned_name': 'User 35 - Xiang Chun Ho, Jason',
  'assigned_email': 'jason.xiang@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 3481',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.27',
  'assigned_name': 'User 27 - Wong Ling Yan, Ada',
  'assigned_email': 'lywong24@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1510',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.5',
  'assigned_name': 'User 5 - Chan Yuet Chi, Jasmine',
  'assigned_email': 'u3633374@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 3325',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.25',
  'assigned_name': 'User 25 - Lee Kai Yan, Alicia',
  'assigned_email': 'kyalicia@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 310',
  'charge_number': 2,
  'defendant_id': 2,
  'assigned_username': 'user.7',
 

In [12]:
pipeline_place_private_vehicle_and_possession = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$unwind": {"path": "$judgement.charges.defendants_of_charge"}
  },
  {
    "$match": {
      "judgement.charges.place_of_offence.nature": "Private vehicle",
      "judgement.charges.defendants_of_charge.trafficking_mode.mode": "Possession for trafficking"
    }
  },
]

data = get_flagged_cases(pipeline_place_private_vehicle_and_possession)
tables["trafficking_mode.7"] = data
data

[{'neutral_citation': '[2021] HKDC 310',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.7',
  'assigned_name': 'User 7 - Liu Chak Yan, Jason',
  'assigned_email': 'u3627300@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 1442',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.13',
  'assigned_name': 'User 13 - Shum Pak Yin',
  'assigned_email': 'pyshum05@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 1441',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.13',
  'assigned_name': 'User 13 - Shum Pak Yin',
  'assigned_email': 'pyshum05@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 780',
  'charge_number': 8,
  'defendant_id': 1,
  'assigned_username': 'user.28',
  'assigned_name': 'User 28 - Wong Tin Wing, Winnie',
  'assigned_email': 'winnie43@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 780',
  'charge_number': 15,
  'defendant_id': 1,
  'assigned_username': 'user.28',
  'assigned_name': 

In [13]:
pipeline_place_street_and_street_level_dealing = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$unwind": {"path": "$judgement.charges.defendants_of_charge"}
  },
  {
    "$match": {
      "judgement.charges.place_of_offence.nature": "Street",
      "judgement.charges.defendants_of_charge.trafficking_mode.mode": "Street-level dealing"
    }
  },
]

data = get_flagged_cases(pipeline_place_street_and_street_level_dealing)
tables["trafficking_mode.8"] = data
data

[{'neutral_citation': '[2021] HKDC 461',
  'charge_number': -1,
  'defendant_id': 1,
  'assigned_username': 'user.23',
  'assigned_name': 'User 23 - Zhou Ziyue, Linda',
  'assigned_email': 'u3612895@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 74',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.29',
  'assigned_name': 'User 29 - Lau Wan',
  'assigned_email': 'u3638683@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 70',
  'charge_number': 4,
  'defendant_id': 1,
  'assigned_username': 'user.33',
  'assigned_name': 'User 33 - Shih Maan Chi, Gigi',
  'assigned_email': 'gigishih@connect.hku.hk'},
 {'neutral_citation': '[2024] HKDC 1803',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.28',
  'assigned_name': 'User 28 - Wong Tin Wing, Winnie',
  'assigned_email': 'winnie43@connect.hku.hk'},
 {'neutral_citation': '[2025] HKDC 165',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.35',
  'assigned_name': 'Us

In [14]:
pipeline_repeated_trafficking_modes = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$unwind": {"path": "$judgement.charges.defendants_of_charge"}
  },
  {
    "$unwind": {"path": "$judgement.charges.defendants_of_charge.trafficking_mode"}
  },
  {
    "$group": {
      "_id": {
        "neutral_citation": "$judgement.neutral_citation",
        "mode": "$judgement.charges.defendants_of_charge.trafficking_mode.mode"
      },
      "charge_numbers": {"$addToSet": "$judgement.charges.charge_no"},
      "verified_by": {"$first": "$verified_by"}
    }
  },
  {
    "$addFields": {
      "charge_count": {"$size": "$charge_numbers"}
    }
  },
  {
    "$match": {
      "charge_count": {"$gte": 2}
    }
  },
  {
    "$lookup": {
      "from": "user",
      "localField": "verified_by",
      "foreignField": "_id",
      "as": "assigned_user"
    }
  },
  {
    "$project": {
      "neutral_citation": "$_id.neutral_citation",
      "trafficking_mode": "$_id.mode",
      "charge_numbers": 1,
      "charge_count": 1,
      "assigned_username": {"$arrayElemAt": ["$assigned_user.username", 0]},
      "assigned_name": {"$arrayElemAt": ["$assigned_user.name", 0]},
      "assigned_email": {"$arrayElemAt": ["$assigned_user.email", 0]},
      "_id": 0
    }
  }
]

data = list(verified_features_collection.aggregate(pipeline_repeated_trafficking_modes))
tables["trafficking_mode.9"] = data
data

[{'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2024] HKDC 2079',
  'trafficking_mode': 'Courier delivery',
  'assigned_username': 'user.28',
  'assigned_name': 'User 28 - Wong Tin Wing, Winnie',
  'assigned_email': 'winnie43@connect.hku.hk'},
 {'charge_numbers': [1, 3],
  'charge_count': 2,
  'neutral_citation': '[2022] HKCFI 205',
  'trafficking_mode': 'Courier delivery',
  'assigned_username': 'user.14',
  'assigned_name': 'User 14 - Mak Ho Lam, Clarence',
  'assigned_email': 'cmak1@connect.hku.hk'},
 {'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2025] HKCFI 934',
  'trafficking_mode': 'Courier delivery',
  'assigned_username': 'user.38',
  'assigned_name': 'User 38 - Wong Tsz Yan, Alyssa',
  'assigned_email': 'alyssaty@connect.hku.hk'},
 {'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2025] HKCFI 209',
  'trafficking_mode': 'Possession for trafficking',
  'assigned_username': 'user.9',
  'assigned_name': 'Us

#### roles(facts)

In [15]:
pipeline_roles_actual_and_support = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$match": {
      "$and": [
        {
          "judgement.charges.defendants_of_charge": {
            "$elemMatch": {"roles_facts.role": "Actual trafficker"}
          }
        },
        {
          "judgement.charges.defendants_of_charge": {
            "$elemMatch": {
              "roles_facts.role": {"$in": ["Courier", "Storekeeper", "Packager"]}
            }
          }
        }
      ]
    }
  },
]

data = get_flagged_cases(pipeline_roles_actual_and_support)
tables["role.1"] = data
data

[{'neutral_citation': '[2021] HKDC 1470',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.40',
  'assigned_name': 'User 40 - Ng Pui Tak, Matthew',
  'assigned_email': 'u3607618@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 785',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.33',
  'assigned_name': 'User 33 - Shih Maan Chi, Gigi',
  'assigned_email': 'gigishih@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 780',
  'charge_number': 8,
  'defendant_id': [1],
  'assigned_username': 'user.28',
  'assigned_name': 'User 28 - Wong Tin Wing, Winnie',
  'assigned_email': 'winnie43@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 780',
  'charge_number': 15,
  'defendant_id': [1],
  'assigned_username': 'user.28',
  'assigned_name': 'User 28 - Wong Tin Wing, Winnie',
  'assigned_email': 'winnie43@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 844',
  'charge_number': 2,
  'defendant_id': [1],
  'assigned_username': 'use

In [16]:
pipeline_cross_border_yes_and_courier_role = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$match": {
      "judgement.charges.cross_border.cross_border": True,
      "judgement.charges.defendants_of_charge": {
        "$elemMatch": {"roles_facts.role": "Courier"}
      }
    }
  },
]

data = get_flagged_cases(pipeline_cross_border_yes_and_courier_role)
tables["roles.2"] = data
data

[{'neutral_citation': '[2021] HKCFI 1536',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.14',
  'assigned_name': 'User 14 - Mak Ho Lam, Clarence',
  'assigned_email': 'cmak1@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 342',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.27',
  'assigned_name': 'User 27 - Wong Ling Yan, Ada',
  'assigned_email': 'lywong24@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2598',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.2',
  'assigned_name': 'User 2 - Wong Hoi Tung, Zoe',
  'assigned_email': 'u3640187@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 1719',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.16',
  'assigned_name': 'User 16 - Lee Hwang, Joseph',
  'assigned_email': 'u3632543@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 1719',
  'charge_number': 2,
  'defendant_id': [1],
  'assigned_username': 'user.16',

In [17]:
pipeline_repeated_roles_facts = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$unwind": {"path": "$judgement.charges.defendants_of_charge"}
  },
  {
    "$unwind": {"path": "$judgement.charges.defendants_of_charge.roles_facts"}
  },
  {
    "$group": {
      "_id": {
        "neutral_citation": "$judgement.neutral_citation",
        "role": "$judgement.charges.defendants_of_charge.roles_facts.role"
      },
      "charge_numbers": {"$addToSet": "$judgement.charges.charge_no"},
      "verified_by": {"$first": "$verified_by"}
    }
  },
  {
    "$addFields": {
      "charge_count": {"$size": "$charge_numbers"}
    }
  },
  {
    "$match": {
      "charge_count": {"$gte": 2}
    }
  },
  {
    "$lookup": {
      "from": "user",
      "localField": "verified_by",
      "foreignField": "_id",
      "as": "assigned_user"
    }
  },
  {
    "$project": {
      "neutral_citation": "$_id.neutral_citation",
      "role": "$_id.role",
      "charge_numbers": 1,
      "charge_count": 1,
      "assigned_username": {"$arrayElemAt": ["$assigned_user.username", 0]},
      "assigned_name": {"$arrayElemAt": ["$assigned_user.name", 0]},
      "assigned_email": {"$arrayElemAt": ["$assigned_user.email", 0]},
      "_id": 0
    }
  }
]

data = list(verified_features_collection.aggregate(pipeline_repeated_roles_facts))
tables["roles.3"] = data
data

[{'charge_numbers': [2, 1],
  'charge_count': 2,
  'neutral_citation': '[2024] HKCFI 2830',
  'role': 'Courier',
  'assigned_username': 'user.19',
  'assigned_name': 'User 19 - Ling Hiu Yi, Flora',
  'assigned_email': 'lhy37@connect.hku.hk'},
 {'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2021] HKCFI 386',
  'role': 'Storekeeper',
  'assigned_username': 'user.3',
  'assigned_name': 'User 3 - Wong Man Hei, Pheobe',
  'assigned_email': 'wongmanheipheobe@connect.hku.hk'},
 {'charge_numbers': [2, 1],
  'charge_count': 2,
  'neutral_citation': '[2025] HKDC 1010',
  'role': 'Packager',
  'assigned_username': 'user.28',
  'assigned_name': 'User 28 - Wong Tin Wing, Winnie',
  'assigned_email': 'winnie43@connect.hku.hk'},
 {'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2024] HKCFI 2956',
  'role': 'Courier',
  'assigned_username': 'user.4',
  'assigned_name': 'User 4 - Chloe Chow',
  'assigned_email': 'chloe1@connect.hku.hk'},
 {'charge_numbers

#### reasons_for_offence

In [18]:
pipeline_reason_addiction_driven = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$match": {
      "judgement.charges.defendants_of_charge": {
        "$elemMatch": {"reasons_for_offence.reason": "Addiction-driven"}
      }
    }
  },
]

data = get_flagged_cases(pipeline_reason_addiction_driven)
tables["reasons_for_offence.1"] = data
data

[{'neutral_citation': '[2025] HKCFI 4288',
  'charge_number': 1,
  'defendant_id': [1, 2],
  'assigned_username': 'super.admin',
  'assigned_name': 'Super Admin',
  'assigned_email': 'admin@admin.com'},
 {'neutral_citation': '[2025] HKCFI 4288',
  'charge_number': 2,
  'defendant_id': [1],
  'assigned_username': 'super.admin',
  'assigned_name': 'Super Admin',
  'assigned_email': 'admin@admin.com'},
 {'neutral_citation': '[2021] HKCFI 1523',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.14',
  'assigned_name': 'User 14 - Mak Ho Lam, Clarence',
  'assigned_email': 'cmak1@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2940',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.11',
  'assigned_name': 'User 11 - Kwai Hoi Yan, Hayley',
  'assigned_email': 'hayleyk@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 393',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.30',
  'assigned_name': 'User 30 - Fung

In [19]:
pipeline_reason_helping_other_people = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$match": {
      "judgement.charges.defendants_of_charge": {
        "$elemMatch": {"reasons_for_offence.reason": "Helping other people"}
      }
    }
  },
]

data = get_flagged_cases(pipeline_reason_helping_other_people)
tables["reasons_for_offence.2"] = data
data

[{'neutral_citation': '[2021] HKDC 1063',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.35',
  'assigned_name': 'User 35 - Xiang Chun Ho, Jason',
  'assigned_email': 'jason.xiang@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1059',
  'charge_number': 2,
  'defendant_id': [2],
  'assigned_username': 'user.34',
  'assigned_name': 'User 34 - Zhang Yuen Ka, Winnie',
  'assigned_email': 'winniezyk@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 785',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.33',
  'assigned_name': 'User 33 - Shih Maan Chi, Gigi',
  'assigned_email': 'gigishih@connect.hku.hk'},
 {'neutral_citation': '[2022] HKCFI 57',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.12',
  'assigned_name': 'User 12 - WU Yangyi',
  'assigned_email': 'u3597419@connect.hku.hk'},
 {'neutral_citation': '[2024] HKDC 1043',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.28',

In [20]:
pipeline_reason_other = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$match": {
      "judgement.charges.defendants_of_charge": {
        "$elemMatch": {"reasons_for_offence.reason": "Other"}
      }
    }
  },
]

data = get_flagged_cases(pipeline_reason_other)
tables["reasons_for_offence.3"] = data
data

[{'neutral_citation': '[2025] HKDC 1630',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'super.admin',
  'assigned_name': 'Super Admin',
  'assigned_email': 'admin@admin.com'},
 {'neutral_citation': '[2025] HKDC 1866',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.41',
  'assigned_name': 'User 41 - WONG, Yi Lok Jason',
  'assigned_email': 'jsonwong@connect.hku.hk'},
 {'neutral_citation': '[2024] HKCFI 675',
  'charge_number': 1,
  'defendant_id': [1, 2],
  'assigned_username': 'user.29',
  'assigned_name': 'User 29 - Lau Wan',
  'assigned_email': 'u3638683@connect.hku.hk'},
 {'neutral_citation': '[2023] HKDC 667',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.41',
  'assigned_name': 'User 41 - WONG, Yi Lok Jason',
  'assigned_email': 'jsonwong@connect.hku.hk'},
 {'neutral_citation': '[2022] HKCFI 674',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.24',
  'assigned_name': 'User 24 -

In [21]:
pipeline_reason_economic_and_financial = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$match": {
      "$and": [
        {
          "judgement.charges.defendants_of_charge": {
            "$elemMatch": {"reasons_for_offence.reason": "Economic hardship"}
          }
        },
        {
          "judgement.charges.defendants_of_charge": {
            "$elemMatch": {"reasons_for_offence.reason": "Financial gain"}
          }
        }
      ]
    }
  },
]

data = get_flagged_cases(pipeline_reason_economic_and_financial)
tables["reasons_for_offence.4"] = data
data

[{'neutral_citation': '[2021] HKCFI 1536',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.14',
  'assigned_name': 'User 14 - Mak Ho Lam, Clarence',
  'assigned_email': 'cmak1@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 420',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.32',
  'assigned_name': 'User 32 - Kulpriya Lam',
  'assigned_email': 'kulpriya@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 420',
  'charge_number': 2,
  'defendant_id': [1],
  'assigned_username': 'user.32',
  'assigned_name': 'User 32 - Kulpriya Lam',
  'assigned_email': 'kulpriya@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 3669',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.28',
  'assigned_name': 'User 28 - Wong Tin Wing, Winnie',
  'assigned_email': 'winnie43@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2777',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.10',
  'ass

#### benefits_received

In [22]:
pipeline_benefits_received_unknown = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$match": {
      "judgement.charges.defendants_of_charge": {
        "$elemMatch": {"benefits_received.received": "Unknown"}
      }
    }
  },
]

data = get_flagged_cases(pipeline_benefits_received_unknown)
tables["benefits_received.1"] = data
data

[{'neutral_citation': '[2021] HKDC 461',
  'charge_number': -1,
  'defendant_id': [1],
  'assigned_username': 'user.23',
  'assigned_name': 'User 23 - Zhou Ziyue, Linda',
  'assigned_email': 'u3612895@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 1441',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.13',
  'assigned_name': 'User 13 - Shum Pak Yin',
  'assigned_email': 'pyshum05@connect.hku.hk'},
 {'neutral_citation': '[2022] HKCFI 205',
  'charge_number': 1,
  'defendant_id': [1, 2],
  'assigned_username': 'user.14',
  'assigned_name': 'User 14 - Mak Ho Lam, Clarence',
  'assigned_email': 'cmak1@connect.hku.hk'},
 {'neutral_citation': '[2022] HKCFI 68',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.11',
  'assigned_name': 'User 11 - Kwai Hoi Yan, Hayley',
  'assigned_email': 'hayleyk@connect.hku.hk'},
 {'neutral_citation': '[2022] HKCFI 68',
  'charge_number': 2,
  'defendant_id': [1],
  'assigned_username': 'user.11',
  'a

In [23]:
pipeline_benefits_received_no_with_values = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$match": {
      "judgement.charges.defendants_of_charge": {
        "$elemMatch": {
          "benefits_received.received": "No",
          "$or": [
            {"$and": [{"benefits_received.amount": {"$ne": None}}, {"benefits_received.amount": {"$exists": True}}]},
            {"$and": [{"benefits_received.amount_type": {"$ne": None}}, {"benefits_received.amount_type": {"$exists": True}}]},
            {"$and": [{"benefits_received.non_monetary_benefits": {"$ne": None}}, {"benefits_received.non_monetary_benefits": {"$exists": True}}]}
          ]
        }
      }
    }
  },
]

data = get_flagged_cases(pipeline_benefits_received_no_with_values)
tables["benefits_received.2"] = data
data

[{'neutral_citation': '[2021] HKCFI 2759',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.1',
  'assigned_name': 'User 1 - Brenda Fung',
  'assigned_email': 'brendafung@connect.hku.hk'},
 {'neutral_citation': '[2022] HKCFI 57',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.12',
  'assigned_name': 'User 12 - WU Yangyi',
  'assigned_email': 'u3597419@connect.hku.hk'},
 {'neutral_citation': '[2025] HKCFI 2744',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.34',
  'assigned_name': 'User 34 - Zhang Yuen Ka, Winnie',
  'assigned_email': 'winniezyk@connect.hku.hk'},
 {'neutral_citation': '[2025] HKCFI 2744',
  'charge_number': 2,
  'defendant_id': [1],
  'assigned_username': 'user.34',
  'assigned_name': 'User 34 - Zhang Yuen Ka, Winnie',
  'assigned_email': 'winniezyk@connect.hku.hk'},
 {'neutral_citation': '[2023] HKDC 1179',
  'charge_number': 1,
  'defendant_id': [1],
  'assigned_username': 'user.14',
  'as

In [24]:
pipeline_repeated_benefits_amount = [
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$unwind": {"path": "$judgement.charges.defendants_of_charge"}
  },
  {
    "$match": {
      "judgement.charges.defendants_of_charge.benefits_received.amount": {"$ne": None}
    }
  },
  {
    "$group": {
      "_id": {
        "neutral_citation": "$judgement.neutral_citation",
        "amount": "$judgement.charges.defendants_of_charge.benefits_received.amount"
      },
      "charge_numbers": {"$addToSet": "$judgement.charges.charge_no"},
      "verified_by": {"$first": "$verified_by"}
    }
  },
  {
    "$addFields": {
      "charge_count": {"$size": "$charge_numbers"}
    }
  },
  {
    "$match": {
      "charge_count": {"$gte": 2}
    }
  },
  {
    "$lookup": {
      "from": "user",
      "localField": "verified_by",
      "foreignField": "_id",
      "as": "assigned_user"
    }
  },
  {
    "$project": {
      "neutral_citation": "$_id.neutral_citation",
      "amount": "$_id.amount",
      "charge_numbers": 1,
      "charge_count": 1,
      "assigned_username": {"$arrayElemAt": ["$assigned_user.username", 0]},
      "assigned_name": {"$arrayElemAt": ["$assigned_user.name", 0]},
      "assigned_email": {"$arrayElemAt": ["$assigned_user.email", 0]},
      "_id": 0
    }
  }
]

data = list(verified_features_collection.aggregate(pipeline_repeated_benefits_amount))
tables["benefits_received.3"] = data
data

[{'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2023] HKDC 205',
  'amount': 250,
  'assigned_username': 'user.38',
  'assigned_name': 'User 38 - Wong Tsz Yan, Alyssa',
  'assigned_email': 'alyssaty@connect.hku.hk'},
 {'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2025] HKCFI 2403',
  'amount': 1000,
  'assigned_username': 'user.36',
  'assigned_name': 'User 36 - Chen Yi To, Louie',
  'assigned_email': 'louiechen@connect.hku.hk'},
 {'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2022] HKCFI 3709',
  'amount': [400, 500],
  'assigned_username': 'user.37',
  'assigned_name': 'User 37 - Lam Pui Man, Linda',
  'assigned_email': 'linda235@connect.hku.hk'},
 {'charge_numbers': [2, 3, 1],
  'charge_count': 3,
  'neutral_citation': '[2024] HKCFI 2968',
  'amount': 40000,
  'assigned_username': 'user.2',
  'assigned_name': 'User 2 - Wong Hoi Tung, Zoe',
  'assigned_email': 'u3640187@connect.hku.hk'},
 {'charge_numbers': [

In [25]:
!pip install openpyxl

In [26]:
def create_excel_file(tables, filename):
	with pd.ExcelWriter(filename, engine="openpyxl") as writer:
		for key, value in tables.items():
			df = pd.DataFrame(value)
			sheetName = ".".join(key.split(".")[-2:])
			df.to_excel(writer, sheet_name=sheetName, index=False)
create_excel_file(tables, "judgement-flagged-cases.xlsx")

## Defendants


In [27]:
defendants_table = {}

#### defendants

In [28]:
pipeline_defendant_nationality_mainland_chinese = [
  {
    "$unwind": {"path": "$defendants.defendants"}
  },
  {
    "$match": {
      "defendants.defendants.nationality.category": "Mainland Chinese"
    }
  },
]

data = get_defendant_flagged_cases(pipeline_defendant_nationality_mainland_chinese)
defendants_table["nationality"] = data
data

[{'neutral_citation': '[2021] HKCFI 1523',
  'defendant_id': 1,
  'defendant_name': '卓節巧',
  'assigned_username': 'user.14',
  'assigned_name': 'User 14 - Mak Ho Lam, Clarence',
  'assigned_email': 'cmak1@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2038',
  'defendant_id': 1,
  'defendant_name': 'CHAN YIP FAI',
  'assigned_username': 'user.19',
  'assigned_name': 'User 19 - Ling Hiu Yi, Flora',
  'assigned_email': 'lhy37@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 3284',
  'defendant_id': 1,
  'defendant_name': '馮梓峰',
  'assigned_username': 'user.24',
  'assigned_name': 'User 24 - Lee Hiu Wa',
  'assigned_email': 'dorcasyy@connect.hku.hk'},
 {'neutral_citation': '[2022] HKCFI 487',
  'defendant_id': 1,
  'defendant_name': '姜春',
  'assigned_username': 'super.admin',
  'assigned_name': 'Super Admin',
  'assigned_email': 'admin@admin.com'},
 {'neutral_citation': '[2025] HKDC 1630',
  'defendant_id': 1,
  'defendant_name': 'NG CHI WAI',
  'assigned_username': 'super.adm

In [29]:
pipeline_defendant_age_missing_one_with_offence_date = [
  {
    "$unwind": {"path": "$defendants.defendants"}
  },
  {
    "$unwind": {"path": "$judgement.charges"}
  },
  {
    "$unwind": {"path": "$judgement.charges.defendants_of_charge"}
  },
  {
    "$match": {
      "judgement.charges.offence_date": {"$ne": None}
    }
  },
  {
    "$match": {
      "$expr": {
        "$eq": [
          "$defendants.defendants.defendant_id",
          "$judgement.charges.defendants_of_charge.defendant_id"
        ]
      }
    }
  },
  {
    "$match": {
      "$or": [
        {
          "$and": [
            {"defendants.defendants.age_at_offence": {"$ne": None}},
            {"defendants.defendants.age_at_sentencing": None}
          ]
        },
        {
          "$and": [
            {"defendants.defendants.age_at_offence": None},
            {"defendants.defendants.age_at_sentencing": {"$ne": None}}
          ]
        }
      ]
    }
  },
]

data = get_defendant_flagged_cases(pipeline_defendant_age_missing_one_with_offence_date)
defendants_table["age_at_offence"] = data
data

[{'neutral_citation': '[2021] HKCFI 1523',
  'defendant_id': 1,
  'defendant_name': '卓節巧',
  'assigned_username': 'user.14',
  'assigned_name': 'User 14 - Mak Ho Lam, Clarence',
  'assigned_email': 'cmak1@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1461',
  'defendant_id': 1,
  'defendant_name': '何皓翔',
  'assigned_username': 'user.39',
  'assigned_name': 'User 39 - Lai Yan Kiu, Ivana',
  'assigned_email': 'u3640281@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 110',
  'defendant_id': 1,
  'defendant_name': '鄺德明',
  'assigned_username': 'user.36',
  'assigned_name': 'User 36 - Chen Yi To, Louie',
  'assigned_email': 'louiechen@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1470',
  'defendant_id': 1,
  'defendant_name': '陳衍麟',
  'assigned_username': 'user.40',
  'assigned_name': 'User 40 - Ng Pui Tak, Matthew',
  'assigned_email': 'u3607618@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1476',
  'defendant_id': 1,
  'defendant_name': '胡少文',
  'assigned_userna

In [30]:
pipeline_defendant_gender_null = [
  {
    "$unwind": {"path": "$defendants.defendants"}
  },
  {
    "$match": {
      "$or": [
        {"defendants.defendants.gender": None},
        {"defendants.defendants.gender": {"$exists": False}}
      ]
    }
  },
]

data = get_defendant_flagged_cases(pipeline_defendant_gender_null)
defendants_table["gender"] = data
data

[{'neutral_citation': '[2021] HKCFI 1536',
  'defendant_id': 1,
  'defendant_name': 'So Tat-keung',
  'assigned_username': 'user.14',
  'assigned_name': 'User 14 - Mak Ho Lam, Clarence',
  'assigned_email': 'cmak1@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2837',
  'defendant_id': 1,
  'defendant_name': '張家彪',
  'assigned_username': 'user.11',
  'assigned_name': 'User 11 - Kwai Hoi Yan, Hayley',
  'assigned_email': 'hayleyk@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 3686',
  'defendant_id': 1,
  'defendant_name': 'Hu Mingjie',
  'assigned_username': 'user.29',
  'assigned_name': 'User 29 - Lau Wan',
  'assigned_email': 'u3638683@connect.hku.hk'},
 {'neutral_citation': '[2022] HKCFI 205',
  'defendant_id': 2,
  'defendant_name': 'Chan Ying-hong',
  'assigned_username': 'user.14',
  'assigned_name': 'User 14 - Mak Ho Lam, Clarence',
  'assigned_email': 'cmak1@connect.hku.hk'},
 {'neutral_citation': '[2023] HKCFI 1150',
  'defendant_id': 1,
  'defendant_name': 'Cheun

In [31]:
pipeline_defendant_parental_custody_selected = [
  {
    "$unwind": {"path": "$defendants.defendants"}
  },
  {
    "$match": {
      "defendants.defendants.parental_status.custody": {"$ne": None}
    }
  },
]

data = get_defendant_flagged_cases(pipeline_defendant_parental_custody_selected)
defendants_table["parental_status"] = data
data

[{'neutral_citation': '[2021] HKDC 1063',
  'defendant_id': 1,
  'defendant_name': '甘梓楊（又名甘國樑）',
  'assigned_username': 'user.35',
  'assigned_name': 'User 35 - Xiang Chun Ho, Jason',
  'assigned_email': 'jason.xiang@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 419',
  'defendant_id': 1,
  'defendant_name': '馬美娟',
  'assigned_username': 'user.32',
  'assigned_name': 'User 32 - Kulpriya Lam',
  'assigned_email': 'kulpriya@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 342',
  'defendant_id': 1,
  'defendant_name': 'Tamayo Gladys Jay',
  'assigned_username': 'user.27',
  'assigned_name': 'User 27 - Wong Ling Yan, Ada',
  'assigned_email': 'lywong24@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 420',
  'defendant_id': 1,
  'defendant_name': '麥子康',
  'assigned_username': 'user.32',
  'assigned_name': 'User 32 - Kulpriya Lam',
  'assigned_email': 'kulpriya@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 3320',
  'defendant_id': 1,
  'defendant_name': 'Ho Lee-ch

In [32]:
pipeline_defendant_drug_treatment_yes = [
  {
    "$unwind": {"path": "$defendants.defendants"}
  },
  {
    "$match": {
      "defendants.defendants.drug_treatment_participation.participated": True
    }
  },
]

data = get_defendant_flagged_cases(pipeline_defendant_drug_treatment_yes)
defendants_table["drug_treatment_participation"] = data
data

[{'neutral_citation': '[2025] HKCFI 4288',
  'defendant_id': 1,
  'defendant_name': 'Yim Kwok-yin, Alex (嚴國賢)',
  'assigned_username': 'super.admin',
  'assigned_name': 'Super Admin',
  'assigned_email': 'admin@admin.com'},
 {'neutral_citation': '[2021] HKDC 1106',
  'defendant_id': 1,
  'defendant_name': '石浩天',
  'assigned_username': 'user.37',
  'assigned_name': 'User 37 - Lam Pui Man, Linda',
  'assigned_email': 'linda235@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1503',
  'defendant_id': 1,
  'defendant_name': '符哲豪',
  'assigned_username': 'user.41',
  'assigned_name': 'User 41 - WONG, Yi Lok Jason',
  'assigned_email': 'jsonwong@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1276',
  'defendant_id': 1,
  'defendant_name': '蘇偉富 SO Wai-fu',
  'assigned_username': 'user.38',
  'assigned_name': 'User 38 - Wong Tsz Yan, Alyssa',
  'assigned_email': 'alyssaty@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1476',
  'defendant_id': 1,
  'defendant_name': '胡少文',
  'as

In [33]:
pipeline_defendant_criminal_records_duplicates = [
  {
    "$unwind": {"path": "$defendants.defendants"}
  },
  {
    "$addFields": {
      "criminal_record_values": {
        "$map": {
          "input": {"$ifNull": ["$defendants.defendants.criminal_records", []]},
          "as": "cr",
          "in": "$$cr.record"
        }
      }
    }
  },
  {
    "$match": {
      "$expr": {
        "$gt": [
          {"$size": "$criminal_record_values"},
          {"$size": {"$setUnion": ["$criminal_record_values", []]}}
        ]
      }
    }
  },
]

data = get_defendant_flagged_cases(pipeline_defendant_criminal_records_duplicates)
defendants_table["criminal_records"] = data
data

[{'neutral_citation': '[2021] HKCFI 1626',
  'defendant_id': 1,
  'defendant_name': 'DINH Ka-yan, Gigi',
  'assigned_username': 'user.15',
  'assigned_name': 'User 15 - Leung Hoi Ching, Julie',
  'assigned_email': 'julie223@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2778',
  'defendant_id': 1,
  'defendant_name': 'Chan Tsz-kin',
  'assigned_username': 'user.14',
  'assigned_name': 'User 14 - Mak Ho Lam, Clarence',
  'assigned_email': 'cmak1@connect.hku.hk'},
 {'neutral_citation': '[2025] HKDC 165',
  'defendant_id': 1,
  'defendant_name': 'SHAIKH SARFARAZ',
  'assigned_username': 'user.35',
  'assigned_name': 'User 35 - Xiang Chun Ho, Jason',
  'assigned_email': 'jason.xiang@connect.hku.hk'},
 {'neutral_citation': '[2023] HKDC 1824',
  'defendant_id': 1,
  'defendant_name': 'LUM WAI MING (ALSO KNOWN AS LAM WAI MING)',
  'assigned_username': 'user.11',
  'assigned_name': 'User 11 - Kwai Hoi Yan, Hayley',
  'assigned_email': 'hayleyk@connect.hku.hk'},
 {'neutral_citation': '[2

In [34]:
pipeline_defendant_positive_habits_rehab = [
  {
    "$unwind": {"path": "$defendants.defendants"}
  },
  {
    "$match": {
      "defendants.defendants.positive_habits_after_arrest": {
        "$elemMatch": {"habit": "Participation in rehabilitation/self-improvement"}
      }
    }
  },
]

data = get_defendant_flagged_cases(pipeline_defendant_positive_habits_rehab)
defendants_table["positive_habits_after_arrest"] = data
data

[{'neutral_citation': '[2021] HKDC 1106',
  'defendant_id': 1,
  'defendant_name': '石浩天',
  'assigned_username': 'user.37',
  'assigned_name': 'User 37 - Lam Pui Man, Linda',
  'assigned_email': 'linda235@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 3481',
  'defendant_id': 1,
  'defendant_name': '黃俊軒',
  'assigned_username': 'user.27',
  'assigned_name': 'User 27 - Wong Ling Yan, Ada',
  'assigned_email': 'lywong24@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 420',
  'defendant_id': 1,
  'defendant_name': '麥子康',
  'assigned_username': 'user.32',
  'assigned_name': 'User 32 - Kulpriya Lam',
  'assigned_email': 'kulpriya@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 3320',
  'defendant_id': 1,
  'defendant_name': 'Ho Lee-chin',
  'assigned_username': 'user.25',
  'assigned_name': 'User 25 - Lee Kai Yan, Alicia',
  'assigned_email': 'kyalicia@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2758',
  'defendant_id': 3,
  'defendant_name': 'Lui Chi-ming',
  '

In [35]:
create_excel_file(defendants_table, "defendant-flagged-cases.xlsx")

## Trials

In [36]:
trials_table = {}

### drugs

In [37]:
pipeline_trials_drug_type_other = [
  {
    "$unwind": {"path": "$trials.trials"}
  },
  {
    "$unwind": {"path": "$trials.trials.drugs"}
  },
  {
    "$match": {
      "trials.trials.drugs.drug_type": "Other"
    }
  },
]

data = get_trials_flagged_cases(pipeline_trials_drug_type_other)
trials_table["drugs.1"] = data
data

[{'neutral_citation': '[2021] HKCFI 2674',
  'charge_number': 2,
  'defendant_id': 1,
  'assigned_username': 'user.23',
  'assigned_name': 'User 23 - Zhou Ziyue, Linda',
  'assigned_email': 'u3612895@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2758',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.1',
  'assigned_name': 'User 1 - Brenda Fung',
  'assigned_email': 'brendafung@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2758',
  'charge_number': 1,
  'defendant_id': 2,
  'assigned_username': 'user.1',
  'assigned_name': 'User 1 - Brenda Fung',
  'assigned_email': 'brendafung@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2758',
  'charge_number': 1,
  'defendant_id': 3,
  'assigned_username': 'user.1',
  'assigned_name': 'User 1 - Brenda Fung',
  'assigned_email': 'brendafung@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 3669',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.28',
  'assigned_name': 'User 2

In [38]:
pipeline_trials_duplicate_drug_type = [
  {
    "$unwind": {"path": "$trials.trials"}
  },
  {
    "$addFields": {
      "drug_types": {
        "$map": {
          "input": {"$ifNull": ["$trials.trials.drugs", []]},
          "as": "drug",
          "in": "$$drug.drug_type"
        }
      }
    }
  },
  {
    "$match": {
      "$expr": {
        "$gt": [
          {"$size": "$drug_types"},
          {"$size": {"$setUnion": ["$drug_types", []]}}
        ]
      }
    }
  },
]

data = get_trials_flagged_cases(pipeline_trials_duplicate_drug_type)
trials_table["drugs.2"] = data
data

[{'neutral_citation': '[2021] HKCFI 3669',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.28',
  'assigned_name': 'User 28 - Wong Tin Wing, Winnie',
  'assigned_email': 'winnie43@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1464',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.4',
  'assigned_name': 'User 4 - Chloe Chow',
  'assigned_email': 'chloe1@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2607',
  'charge_number': 2,
  'defendant_id': 1,
  'assigned_username': 'user.20',
  'assigned_name': 'User 20 - Cheung Ka-Lun, Karen',
  'assigned_email': 'karen.c13@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2615',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.21',
  'assigned_name': 'User 21 - CHOI Hoi Yeung, Shannon',
  'assigned_email': 'chys@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2616',
  'charge_number': 2,
  'defendant_id': 1,
  'assigned_username': 'user.21',
  'assigne

In [39]:
pipeline_trials_drug_quantity_zero_or_null = [
  {
    "$unwind": {"path": "$trials.trials"}
  },
  {
    "$unwind": {"path": "$trials.trials.drugs"}
  },
  {
    "$match": {
      "$or": [
        {"trials.trials.drugs.quantity": {"$in": [0, None]}},
        {"trials.trials.drugs.quantity": {"$exists": False}}
      ]
    }
  },
]

data = get_trials_flagged_cases(pipeline_trials_drug_quantity_zero_or_null)
trials_table["drugs.3"] = data
data

[{'neutral_citation': '[2021] HKDC 453',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.6',
  'assigned_name': 'User 6 - Sung Yan Ling',
  'assigned_email': 'u3640306@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 795',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.26',
  'assigned_name': 'User 26 - Ng Kin Chun, Ken',
  'assigned_email': 'ken.ngkinchun@connect.hku.hk'},
 {'neutral_citation': '[2023] HKDC 1852',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.14',
  'assigned_name': 'User 14 - Mak Ho Lam, Clarence',
  'assigned_email': 'cmak1@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1452',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.19',
  'assigned_name': 'User 19 - Ling Hiu Yi, Flora',
  'assigned_email': 'lhy37@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 3918',
  'charge_number': 2,
  'defendant_id': 1,
  'assigned_username': 'user.19',
  'assigned_name':

In [40]:
pipeline_trials_repeated_drug_type_quantity = [
  {
    "$unwind": {"path": "$trials.trials"}
  },
  {
    "$unwind": {"path": "$trials.trials.drugs"}
  },
  {
    "$group": {
      "_id": {
        "neutral_citation": "$judgement.neutral_citation",
        "drug_type": "$trials.trials.drugs.drug_type",
        "quantity": "$trials.trials.drugs.quantity"
      },
      "charge_numbers": {"$addToSet": "$trials.trials.charge_type.charge_no"},
      "verified_by": {"$first": "$verified_by"}
    }
  },
  {
    "$addFields": {
      "charge_count": {"$size": "$charge_numbers"}
    }
  },
  {
    "$match": {
      "charge_count": {"$gte": 2}
    }
  },
  {
    "$lookup": {
      "from": "user",
      "localField": "verified_by",
      "foreignField": "_id",
      "as": "assigned_user"
    }
  },
  {
    "$project": {
      "neutral_citation": "$_id.neutral_citation",
      "drug_type": "$_id.drug_type",
      "quantity": "$_id.quantity",
      "charge_numbers": 1,
      "charge_count": 1,
      "assigned_username": {"$arrayElemAt": ["$assigned_user.username", 0]},
      "assigned_name": {"$arrayElemAt": ["$assigned_user.name", 0]},
      "assigned_email": {"$arrayElemAt": ["$assigned_user.email", 0]},
      "_id": 0
    }
  }
]

data = list(verified_features_collection.aggregate(pipeline_trials_repeated_drug_type_quantity))
trials_table["drugs.4"] = data
data

[{'charge_numbers': [2, 1],
  'charge_count': 2,
  'neutral_citation': '[2023] HKCFI 2275',
  'drug_type': 'Cocaine',
  'quantity': 3025,
  'assigned_username': 'user.23',
  'assigned_name': 'User 23 - Zhou Ziyue, Linda',
  'assigned_email': 'u3612895@connect.hku.hk'},
 {'charge_numbers': [2, 1],
  'charge_count': 2,
  'neutral_citation': '[2025] HKCFI 3617',
  'drug_type': 'Fluorodeschloroketamine',
  'quantity': 0,
  'assigned_username': 'user.5',
  'assigned_name': 'User 5 - Chan Yuet Chi, Jasmine',
  'assigned_email': 'u3633374@connect.hku.hk'},
 {'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2024] HKCFI 3176',
  'drug_type': 'Cocaine',
  'quantity': 1112,
  'assigned_username': 'user.19',
  'assigned_name': 'User 19 - Ling Hiu Yi, Flora',
  'assigned_email': 'lhy37@connect.hku.hk'},
 {'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2021] HKCFI 3324',
  'drug_type': 'Ketamine',
  'quantity': 3792.9},
 {'charge_numbers': [1, 2],
  'cha

### aggravating_factors

In [41]:
pipeline_trials_aggravating_other = [
  {
    "$unwind": {"path": "$trials.trials"}
  },
  {
    "$unwind": {"path": "$trials.trials.aggravating_factors"}
  },
  {
    "$match": {
      "trials.trials.aggravating_factors.factor": "Other"
    }
  },
]

data = get_trials_flagged_cases(pipeline_trials_aggravating_other)
trials_table["aggravating_factors.1"] = data
data

[{'neutral_citation': '[2021] HKCFI 1718',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.16',
  'assigned_name': 'User 16 - Lee Hwang, Joseph',
  'assigned_email': 'u3632543@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1503',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.41',
  'assigned_name': 'User 41 - WONG, Yi Lok Jason',
  'assigned_email': 'jsonwong@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 306',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.7',
  'assigned_name': 'User 7 - Liu Chak Yan, Jason',
  'assigned_email': 'u3627300@connect.hku.hk'},
 {'neutral_citation': '[2025] HKDC 1630',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'super.admin',
  'assigned_name': 'Super Admin',
  'assigned_email': 'admin@admin.com'},
 {'neutral_citation': '[2021] HKDC 800',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.25',
  'assigned_name': 'User 25 - Lee

In [42]:
pipeline_trials_aggravating_missing_enhancement = [
  {
    "$unwind": {"path": "$trials.trials"}
  },
  {
    "$unwind": {"path": "$trials.trials.aggravating_factors"}
  },
  {
    "$match": {
      "$or": [
        {"trials.trials.aggravating_factors.enhancement_months": {"$in": [None, 0]}},
        {"trials.trials.aggravating_factors.enhancement_months": {"$exists": False}}
      ]
    }
  },
]

data = get_trials_flagged_cases(pipeline_trials_aggravating_missing_enhancement)
trials_table["aggravating_factors.2"] = data
data

[{'neutral_citation': '[2021] HKCFI 393',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.30',
  'assigned_name': 'User 30 - Fung Chun Hei, Justin',
  'assigned_email': 'justinf@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1063',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.35',
  'assigned_name': 'User 35 - Xiang Chun Ho, Jason',
  'assigned_email': 'jason.xiang@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2600',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.2',
  'assigned_name': 'User 2 - Wong Hoi Tung, Zoe',
  'assigned_email': 'u3640187@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 1719',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.16',
  'assigned_name': 'User 16 - Lee Hwang, Joseph',
  'assigned_email': 'u3632543@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 1719',
  'charge_number': 2,
  'defendant_id': 1,
  'assigned_username': 'user.16',
  

In [43]:
pipeline_trials_repeated_aggravating_factors = [
  {
    "$unwind": {"path": "$trials.trials"}
  },
  {
    "$unwind": {"path": "$trials.trials.aggravating_factors"}
  },
  {
    "$group": {
      "_id": {
        "neutral_citation": "$judgement.neutral_citation",
        "factor": "$trials.trials.aggravating_factors.factor"
      },
      "charge_numbers": {"$addToSet": "$trials.trials.charge_type.charge_no"},
      "verified_by": {"$first": "$verified_by"}
    }
  },
  {
    "$addFields": {
      "charge_count": {"$size": "$charge_numbers"}
    }
  },
  {
    "$match": {
      "charge_count": {"$gte": 2}
    }
  },
  {
    "$lookup": {
      "from": "user",
      "localField": "verified_by",
      "foreignField": "_id",
      "as": "assigned_user"
    }
  },
  {
    "$project": {
      "neutral_citation": "$_id.neutral_citation",
      "factor": "$_id.factor",
      "charge_numbers": 1,
      "charge_count": 1,
      "assigned_username": {"$arrayElemAt": ["$assigned_user.username", 0]},
      "assigned_name": {"$arrayElemAt": ["$assigned_user.name", 0]},
      "assigned_email": {"$arrayElemAt": ["$assigned_user.email", 0]},
      "_id": 0
    }
  }
]

data = list(verified_features_collection.aggregate(pipeline_trials_repeated_aggravating_factors))
trials_table["aggravating_factors.3"] = data
data

[{'charge_numbers': [2, 1],
  'charge_count': 2,
  'neutral_citation': '[2024] HKCFI 3403',
  'factor': 'Multiple drugs',
  'assigned_username': 'user.34',
  'assigned_name': 'User 34 - Zhang Yuen Ka, Winnie',
  'assigned_email': 'winniezyk@connect.hku.hk'},
 {'charge_numbers': [2, 1],
  'charge_count': 2,
  'neutral_citation': '[2021] HKCFI 494',
  'factor': 'Multiple drugs',
  'assigned_username': 'user.15',
  'assigned_name': 'User 15 - Leung Hoi Ching, Julie',
  'assigned_email': 'julie223@connect.hku.hk'},
 {'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2025] HKCFI 3617',
  'factor': 'Multiple drugs',
  'assigned_username': 'user.5',
  'assigned_name': 'User 5 - Chan Yuet Chi, Jasmine',
  'assigned_email': 'u3633374@connect.hku.hk'},
 {'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2022] HKDC 1191',
  'factor': 'Other',
  'assigned_username': 'user.23',
  'assigned_name': 'User 23 - Zhou Ziyue, Linda',
  'assigned_email': 'u3612895@

### mitigating_factors

In [44]:
pipeline_trials_mitigating_assistance = [
  {
    "$unwind": {"path": "$trials.trials"}
  },
  {
    "$unwind": {"path": "$trials.trials.mitigating_factors"}
  },
  {
    "$match": {
      "trials.trials.mitigating_factors.factor": {
        "$in": [
          "Assistance - limited",
          "Assistance - useful",
          "Assistance - testify",
          "Assistance - risk"
        ]
      }
    }
  },
]

data = get_trials_flagged_cases(pipeline_trials_mitigating_assistance)
trials_table["mitigating_factors.1"] = data
data

[{'neutral_citation': '[2021] HKCFI 1536',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.14',
  'assigned_name': 'User 14 - Mak Ho Lam, Clarence',
  'assigned_email': 'cmak1@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2598',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.2',
  'assigned_name': 'User 2 - Wong Hoi Tung, Zoe',
  'assigned_email': 'u3640187@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2600',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.2',
  'assigned_name': 'User 2 - Wong Hoi Tung, Zoe',
  'assigned_email': 'u3640187@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1461',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.39',
  'assigned_name': 'User 39 - Lai Yan Kiu, Ivana',
  'assigned_email': 'u3640281@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2372',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.19',
  'assigne

In [45]:
pipeline_trials_mitigating_rehabilitation = [
  {
    "$unwind": {"path": "$trials.trials"}
  },
  {
    "$unwind": {"path": "$trials.trials.mitigating_factors"}
  },
  {
    "$match": {
      "trials.trials.mitigating_factors.factor": "Rehabilitation programme"
    }
  },
]

data = get_trials_flagged_cases(pipeline_trials_mitigating_rehabilitation)
trials_table["mitigating_factors.2"] = data
data

[{'neutral_citation': '[2021] HKDC 1106',
  'charge_number': 3,
  'defendant_id': 1,
  'assigned_username': 'user.37',
  'assigned_name': 'User 37 - Lam Pui Man, Linda',
  'assigned_email': 'linda235@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 1720',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.17',
  'assigned_name': 'User 17 - He Fangyi, Joanna',
  'assigned_email': 'hphiii3355@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 3733',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.3',
  'assigned_name': 'User 3 - Wong Man Hei, Pheobe',
  'assigned_email': 'wongmanheipheobe@connect.hku.hk'},
 {'neutral_citation': '[2022] HKCFI 297',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.14',
  'assigned_name': 'User 14 - Mak Ho Lam, Clarence',
  'assigned_email': 'cmak1@connect.hku.hk'},
 {'neutral_citation': '[2022] HKCFI 183',
  'charge_number': 3,
  'defendant_id': 1,
  'assigned_username': 'user.15',

In [46]:
pipeline_trials_mitigating_other = [
  {
    "$unwind": {"path": "$trials.trials"}
  },
  {
    "$unwind": {"path": "$trials.trials.mitigating_factors"}
  },
  {
    "$match": {
      "trials.trials.mitigating_factors.factor": "Other"
    }
  },
]

data = get_trials_flagged_cases(pipeline_trials_mitigating_other)
trials_table["mitigating_factors.3"] = data
data

[{'neutral_citation': '[2021] HKDC 461',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.23',
  'assigned_name': 'User 23 - Zhou Ziyue, Linda',
  'assigned_email': 'u3612895@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2940',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.11',
  'assigned_name': 'User 11 - Kwai Hoi Yan, Hayley',
  'assigned_email': 'hayleyk@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1063',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.35',
  'assigned_name': 'User 35 - Xiang Chun Ho, Jason',
  'assigned_email': 'jason.xiang@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2600',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.2',
  'assigned_name': 'User 2 - Wong Hoi Tung, Zoe',
  'assigned_email': 'u3640187@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1089',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.36',
  'as

In [47]:
pipeline_trials_mitigating_missing_reduction = [
  {
    "$unwind": {"path": "$trials.trials"}
  },
  {
    "$unwind": {"path": "$trials.trials.mitigating_factors"}
  },
  {
    "$match": {
      "$and": [
        {
          "$or": [
            {"trials.trials.mitigating_factors.reduction_months": {"$in": [None, 0]}},
            {"trials.trials.mitigating_factors.reduction_months": {"$exists": False}}
          ]
        },
        {
          "$or": [
            {"trials.trials.mitigating_factors.reduction_percentage": {"$in": [None, 0]}},
            {"trials.trials.mitigating_factors.reduction_percentage": {"$exists": False}}
          ]
        }
      ]
    }
  },
]

data = get_trials_flagged_cases(pipeline_trials_mitigating_missing_reduction)
trials_table["mitigating_factors.4"] = data
data

[{'neutral_citation': '[2021] HKCFI 1536',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.14',
  'assigned_name': 'User 14 - Mak Ho Lam, Clarence',
  'assigned_email': 'cmak1@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1063',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.35',
  'assigned_name': 'User 35 - Xiang Chun Ho, Jason',
  'assigned_email': 'jason.xiang@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1063',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.35',
  'assigned_name': 'User 35 - Xiang Chun Ho, Jason',
  'assigned_email': 'jason.xiang@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1106',
  'charge_number': 3,
  'defendant_id': 1,
  'assigned_username': 'user.37',
  'assigned_name': 'User 37 - Lam Pui Man, Linda',
  'assigned_email': 'linda235@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1106',
  'charge_number': 3,
  'defendant_id': 1,
  'assigned_username': 'user.37',

### guilty_plea

In [48]:
pipeline_trials_guilty_plea_reduction_unknown_stage = [
  {
    "$unwind": {"path": "$trials.trials"}
  },
  {
    "$match": {
      "trials.trials.guilty_plea.reduction_percentage": {"$ne": None, "$exists": True}
    }
  },
  {
    "$match": {
      "$or": [
        {"trials.trials.guilty_plea.high_court_stage": "Unknown"},
        {"trials.trials.guilty_plea.district_court_stage": "Unknown"}
      ]
    }
  },
]

data = get_trials_flagged_cases(pipeline_trials_guilty_plea_reduction_unknown_stage)
trials_table["guilty_plea.1"] = data
data

[{'neutral_citation': '[2021] HKDC 1075',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.35',
  'assigned_name': 'User 35 - Xiang Chun Ho, Jason',
  'assigned_email': 'jason.xiang@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1106',
  'charge_number': 3,
  'defendant_id': 1,
  'assigned_username': 'user.37',
  'assigned_name': 'User 37 - Lam Pui Man, Linda',
  'assigned_email': 'linda235@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 339',
  'charge_number': 1,
  'defendant_id': 2,
  'assigned_username': 'user.9',
  'assigned_name': 'User 9 - Xu Wenxuan',
  'assigned_email': 'jonwardxu@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1510',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.5',
  'assigned_name': 'User 5 - Chan Yuet Chi, Jasmine',
  'assigned_email': 'u3633374@connect.hku.hk'},
 {'neutral_citation': '[2021] HKDC 1514',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.5',
  'assigned_n

### sentencing
- Manual check: cross-check against guidelines/model for outliers.

In [49]:
# Current model inputs:

# self_consume = x[0] -> trials.trials[].mitigating_factors[].factor = 'Self-consumption'
# assist_authorities = x[1] -> trials.trials[].mitigating_factors[].factor = 'Assistance - limited' | 'Assistance - useful' | 'Assistance - testify' | 'Assistance - risk'
# other_mitigating = x[2] -> trials.trials[].mitigating_factors[].factor = 'Other'
# refugee = x[3]
# bail = x[4]
# persistent = x[5]
# international = x[6]
# cocaine_amount = x[7]
# heroin_amount = x[8]
# meth_amount = x[9]
# ketamine_amount = x[10]
# nimetazepam_amount = x[11]
# ecstasy_amount = x[12]
# cannabisresin_amount = x[13]
# herbalcannabis_amount = x[14]
# plea_early = x[15]
# plea_late = x[16]

In [50]:
pipeline_trials_notional_sentence_mismatch = [
  {
    "$unwind": {"path": "$trials.trials"}
  },
  {
    "$addFields": {
		"starting_point_months": "$trials.trials.starting_point.total_months",
		"sentence_after_role_months": "$trials.trials.sentence_after_role.total_months",
		"notional_sentence_months": "$trials.trials.notional_sentence.total_months",
		"enhancement_months_total": {
			"$sum": {
				"$map": {
					"input": {"$ifNull": ["$trials.trials.aggravating_factors", []]},
					"as": "af",
					"in": {"$ifNull": ["$$af.enhancement_months", 0]}
				}
			}
		}
	}
  },
  {
    "$addFields": {
      "base_months": {"$ifNull": ["$sentence_after_role_months", "$starting_point_months"]},
	}
  },
  {
    "$addFields": {
      "expected_notional_months": {"$add": ["$base_months", "$enhancement_months_total"]}
    }
  },
  {
    "$match": {
      "$expr": {"$ne": ["$notional_sentence_months", "$expected_notional_months"]}
    }
  }
]

data = get_trials_flagged_cases(pipeline_trials_notional_sentence_mismatch)
# data = list(verified_features_collection.aggregate(pipeline_trials_notional_sentence_mismatch))
trials_table["sentencing.2"] = data
data

[{'neutral_citation': '[2021] HKCFI 393',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.30',
  'assigned_name': 'User 30 - Fung Chun Hei, Justin',
  'assigned_email': 'justinf@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 3684',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.29',
  'assigned_name': 'User 29 - Lau Wan',
  'assigned_email': 'u3638683@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 3320',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.25',
  'assigned_name': 'User 25 - Lee Kai Yan, Alicia',
  'assigned_email': 'kyalicia@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 1726',
  'charge_number': 1,
  'defendant_id': 1,
  'assigned_username': 'user.17',
  'assigned_name': 'User 17 - He Fangyi, Joanna',
  'assigned_email': 'hphiii3355@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 1726',
  'charge_number': 2,
  'defendant_id': 1,
  'assigned_username': 'user.17',
  'assigned_

In [51]:
pipeline_trials_repeated_starting_point = [
  {
    "$unwind": {"path": "$trials.trials"}
  },
  {
    "$addFields": {
      "starting_point_months": {
        "$ifNull": [
          "$trials.trials.starting_point.total_months",
          {
            "$add": [
              {"$multiply": ["$trials.trials.starting_point.sentence_years", 12]},
              "$trials.trials.starting_point.sentence_months"
            ]
          }
        ]
      }
    }
  },
  {
    "$group": {
      "_id": {
        "neutral_citation": "$judgement.neutral_citation",
        "starting_point_months": "$starting_point_months"
      },
      "charge_numbers": {"$addToSet": "$trials.trials.charge_type.charge_no"},
      "verified_by": {"$first": "$verified_by"}
    }
  },
  {
    "$addFields": {
      "charge_count": {"$size": "$charge_numbers"}
    }
  },
  {
    "$match": {
      "charge_count": {"$gte": 2}
    }
  },
  {
    "$lookup": {
      "from": "user",
      "localField": "verified_by",
      "foreignField": "_id",
      "as": "assigned_user"
    }
  },
  {
    "$project": {
      "neutral_citation": "$_id.neutral_citation",
      "starting_point_months": "$_id.starting_point_months",
      "charge_numbers": 1,
      "charge_count": 1,
      "assigned_username": {"$arrayElemAt": ["$assigned_user.username", 0]},
      "assigned_name": {"$arrayElemAt": ["$assigned_user.name", 0]},
      "assigned_email": {"$arrayElemAt": ["$assigned_user.email", 0]},
      "_id": 0
    }
  }
]

data = list(verified_features_collection.aggregate(pipeline_trials_repeated_starting_point))
trials_table["sentencing.3"] = data
data

[{'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2025] HKCFI 4057',
  'starting_point_months': None,
  'assigned_username': 'user.35',
  'assigned_name': 'User 35 - Xiang Chun Ho, Jason',
  'assigned_email': 'jason.xiang@connect.hku.hk'},
 {'charge_numbers': [3, 5, 1, 2, 6, 4],
  'charge_count': 6,
  'neutral_citation': '[2023] HKDC 1827',
  'starting_point_months': None,
  'assigned_username': 'user.9',
  'assigned_name': 'User 9 - Xu Wenxuan',
  'assigned_email': 'jonwardxu@connect.hku.hk'},
 {'charge_numbers': [2, 1],
  'charge_count': 2,
  'neutral_citation': '[2024] HKCFI 2543',
  'starting_point_months': None,
  'assigned_username': 'user.4',
  'assigned_name': 'User 4 - Chloe Chow',
  'assigned_email': 'chloe1@connect.hku.hk'},
 {'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2025] HKCFI 1432',
  'starting_point_months': 265,
  'assigned_username': 'user.16',
  'assigned_name': 'User 16 - Lee Hwang, Joseph',
  'assigned_email': 'u3

In [52]:
pipeline_trials_repeated_sentence_after_role = [
  {
    "$unwind": {"path": "$trials.trials"}
  },
  {
    "$match": {
      "trials.trials.sentence_after_role": {"$ne": None}
    }
  },
  {
    "$addFields": {
      "sentence_after_role_months": {
        "$ifNull": [
          "$trials.trials.sentence_after_role.total_months",
          {
            "$add": [
              {"$multiply": ["$trials.trials.sentence_after_role.sentence_years", 12]},
              "$trials.trials.sentence_after_role.sentence_months"
            ]
          }
        ]
      }
    }
  },
  {
    "$group": {
      "_id": {
        "neutral_citation": "$judgement.neutral_citation",
        "sentence_after_role_months": "$sentence_after_role_months"
      },
      "charge_numbers": {"$addToSet": "$trials.trials.charge_type.charge_no"},
      "verified_by": {"$first": "$verified_by"}
    }
  },
  {
    "$addFields": {
      "charge_count": {"$size": "$charge_numbers"}
    }
  },
  {
    "$match": {
      "charge_count": {"$gte": 2}
    }
  },
  {
    "$lookup": {
      "from": "user",
      "localField": "verified_by",
      "foreignField": "_id",
      "as": "assigned_user"
    }
  },
  {
    "$project": {
      "neutral_citation": "$_id.neutral_citation",
      "sentence_after_role_months": "$_id.sentence_after_role_months",
      "charge_numbers": 1,
      "charge_count": 1,
      "assigned_username": {"$arrayElemAt": ["$assigned_user.username", 0]},
      "assigned_name": {"$arrayElemAt": ["$assigned_user.name", 0]},
      "assigned_email": {"$arrayElemAt": ["$assigned_user.email", 0]},
      "_id": 0
    }
  }
]

data = list(verified_features_collection.aggregate(pipeline_trials_repeated_sentence_after_role))
trials_table["sentencing.4"] = data
data

[{'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2022] HKDC 1191',
  'sentence_after_role_months': 12,
  'assigned_username': 'user.23',
  'assigned_name': 'User 23 - Zhou Ziyue, Linda',
  'assigned_email': 'u3612895@connect.hku.hk'},
 {'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2025] HKCFI 2744',
  'sentence_after_role_months': 384,
  'assigned_username': 'user.34',
  'assigned_name': 'User 34 - Zhang Yuen Ka, Winnie',
  'assigned_email': 'winniezyk@connect.hku.hk'},
 {'charge_numbers': [3, 4],
  'charge_count': 2,
  'neutral_citation': '[2024] HKCFI 595',
  'sentence_after_role_months': 300,
  'assigned_username': 'user.26',
  'assigned_name': 'User 26 - Ng Kin Chun, Ken',
  'assigned_email': 'ken.ngkinchun@connect.hku.hk'},
 {'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2024] HKCFI 2417',
  'sentence_after_role_months': 192,
  'assigned_username': 'user.27',
  'assigned_name': 'User 27 - Wong Ling Yan, Ada

In [53]:
pipeline_trials_repeated_notional_sentence = [
  {
    "$unwind": {"path": "$trials.trials"}
  },
  {
    "$addFields": {
      "notional_sentence_months": {
        "$ifNull": [
          "$trials.trials.notional_sentence.total_months",
          {
            "$add": [
              {"$multiply": ["$trials.trials.notional_sentence.sentence_years", 12]},
              "$trials.trials.notional_sentence.sentence_months"
            ]
          }
        ]
      }
    }
  },
  {
    "$group": {
      "_id": {
        "neutral_citation": "$judgement.neutral_citation",
        "notional_sentence_months": "$notional_sentence_months"
      },
      "charge_numbers": {"$addToSet": "$trials.trials.charge_type.charge_no"},
      "verified_by": {"$first": "$verified_by"}
    }
  },
  {
    "$addFields": {
      "charge_count": {"$size": "$charge_numbers"}
    }
  },
  {
    "$match": {
      "charge_count": {"$gte": 2}
    }
  },
  {
    "$lookup": {
      "from": "user",
      "localField": "verified_by",
      "foreignField": "_id",
      "as": "assigned_user"
    }
  },
  {
    "$project": {
      "neutral_citation": "$_id.neutral_citation",
      "notional_sentence_months": "$_id.notional_sentence_months",
      "charge_numbers": 1,
      "charge_count": 1,
      "assigned_username": {"$arrayElemAt": ["$assigned_user.username", 0]},
      "assigned_name": {"$arrayElemAt": ["$assigned_user.name", 0]},
      "assigned_email": {"$arrayElemAt": ["$assigned_user.email", 0]},
      "_id": 0
    }
  }
]

data = list(verified_features_collection.aggregate(pipeline_trials_repeated_notional_sentence))
trials_table["sentencing.5"] = data
data

[{'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2023] HKCFI 943',
  'notional_sentence_months': 144,
  'assigned_username': 'user.10',
  'assigned_name': 'User 10 - Ma Yin Ka, Otto',
  'assigned_email': 'ottomyk@connect.hku.hk'},
 {'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2025] HKCFI 1709',
  'notional_sentence_months': 150,
  'assigned_username': 'user.16',
  'assigned_name': 'User 16 - Lee Hwang, Joseph',
  'assigned_email': 'u3632543@connect.hku.hk'},
 {'charge_numbers': [1, 3],
  'charge_count': 2,
  'neutral_citation': '[2022] HKDC 243',
  'notional_sentence_months': 24,
  'assigned_username': 'user.5',
  'assigned_name': 'User 5 - Chan Yuet Chi, Jasmine',
  'assigned_email': 'u3633374@connect.hku.hk'},
 {'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2024] HKCFI 160',
  'notional_sentence_months': 164,
  'assigned_username': 'user.8',
  'assigned_name': 'User 8 - Zou Jincheng, Zoey',
  'assigned_email'

In [54]:
pipeline_trials_repeated_final_sentence = [
  {
    "$unwind": {"path": "$trials.trials"}
  },
  {
    "$addFields": {
      "final_sentence_months": {
        "$ifNull": [
          "$trials.trials.final_sentence.total_months",
          {
            "$add": [
              {"$multiply": ["$trials.trials.final_sentence.sentence_years", 12]},
              "$trials.trials.final_sentence.sentence_months"
            ]
          }
        ]
      }
    }
  },
  {
    "$group": {
      "_id": {
        "neutral_citation": "$judgement.neutral_citation",
        "final_sentence_months": "$final_sentence_months"
      },
      "charge_numbers": {"$addToSet": "$trials.trials.charge_type.charge_no"},
      "verified_by": {"$first": "$verified_by"}
    }
  },
  {
    "$addFields": {
      "charge_count": {"$size": "$charge_numbers"}
    }
  },
  {
    "$match": {
      "charge_count": {"$gte": 2}
    }
  },
  {
    "$lookup": {
      "from": "user",
      "localField": "verified_by",
      "foreignField": "_id",
      "as": "assigned_user"
    }
  },
  {
    "$project": {
      "neutral_citation": "$_id.neutral_citation",
      "final_sentence_months": "$_id.final_sentence_months",
      "charge_numbers": 1,
      "charge_count": 1,
      "assigned_username": {"$arrayElemAt": ["$assigned_user.username", 0]},
      "assigned_name": {"$arrayElemAt": ["$assigned_user.name", 0]},
      "assigned_email": {"$arrayElemAt": ["$assigned_user.email", 0]},
      "_id": 0
    }
  }
]

data = list(verified_features_collection.aggregate(pipeline_trials_repeated_final_sentence))
trials_table["sentencing.6"] = data
data

[{'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2025] HKCFI 3683',
  'final_sentence_months': 140,
  'assigned_username': 'user.17',
  'assigned_name': 'User 17 - He Fangyi, Joanna',
  'assigned_email': 'hphiii3355@connect.hku.hk'},
 {'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2024] HKCFI 2968',
  'final_sentence_months': 66,
  'assigned_username': 'user.2',
  'assigned_name': 'User 2 - Wong Hoi Tung, Zoe',
  'assigned_email': 'u3640187@connect.hku.hk'},
 {'charge_numbers': [2, 1],
  'charge_count': 2,
  'neutral_citation': '[2025] HKCFI 2560',
  'final_sentence_months': 176,
  'assigned_username': 'user.35',
  'assigned_name': 'User 35 - Xiang Chun Ho, Jason',
  'assigned_email': 'jason.xiang@connect.hku.hk'},
 {'charge_numbers': [1, 2],
  'charge_count': 2,
  'neutral_citation': '[2024] HKCFI 1707',
  'final_sentence_months': 56,
  'assigned_username': 'user.4',
  'assigned_name': 'User 4 - Chloe Chow',
  'assigned_email': 'chloe1@

In [55]:
create_excel_file(trials_table, "trials-flagged-cases.xlsx")

## Excluded from training

In [56]:
pipeline_excluded = [
  {
    "$match": {
      "exclude": True
    }
  },
  {
    "$lookup": {
      "from": "user",
      "localField": "verified_by",
      "foreignField": "_id",
      "as": "assigned_user"
    }
  },
  {
    "$project": {
      "neutral_citation": "$judgement.neutral_citation",
      "assigned_username": {"$arrayElemAt": ["$assigned_user.username", 0]},
      "assigned_name": {"$arrayElemAt": ["$assigned_user.name", 0]},
      "assigned_email": {"$arrayElemAt": ["$assigned_user.email", 0]},
      "_id": 0
    }
  }
]

excluded_data = list(verified_features_collection.aggregate(pipeline_excluded))
pd.DataFrame(excluded_data).to_excel("excluded-flagged-cases.xlsx", index=False)
excluded_data


[{'neutral_citation': '[2021] HKDC 1503',
  'assigned_username': 'user.41',
  'assigned_name': 'User 41 - WONG, Yi Lok Jason',
  'assigned_email': 'jsonwong@connect.hku.hk'},
 {'neutral_citation': '[2021] HKCFI 2607',
  'assigned_username': 'user.20',
  'assigned_name': 'User 20 - Cheung Ka-Lun, Karen',
  'assigned_email': 'karen.c13@connect.hku.hk'},
 {'neutral_citation': '[2022] HKCFI 3033',
  'assigned_username': 'super.admin',
  'assigned_name': 'Super Admin',
  'assigned_email': 'admin@admin.com'},
 {'neutral_citation': '[2022] HKCFI 487',
  'assigned_username': 'super.admin',
  'assigned_name': 'Super Admin',
  'assigned_email': 'admin@admin.com'},
 {'neutral_citation': '[2025] HKDC 1630',
  'assigned_username': 'super.admin',
  'assigned_name': 'Super Admin',
  'assigned_email': 'admin@admin.com'},
 {'neutral_citation': '[2021] HKDC 780',
  'assigned_username': 'user.28',
  'assigned_name': 'User 28 - Wong Tin Wing, Winnie',
  'assigned_email': 'winnie43@connect.hku.hk'},
 {'neu